# Day 40 — A Hundred Opinions

## Random Forests

**Boston University · Data Science Program**

---

Yesterday's decision tree was the first model in this course you could read out loud.
It was also the twitchiest: drop a random 10% of the training rows, refit, and a question
inside the tree moves.

Today we fix that, with an idea so simple it feels like cheating:

> **If one tree is a coin-flip away from a different answer, grow a hundred of them on
> slightly different data and count the votes.**

### What you will be able to do by the end

| | |
|---|---|
| 1 | Explain bootstrap sampling and why each tree sees only ~63% of the rows |
| 2 | Fit a random forest and say why it needs almost no tuning |
| 3 | Use the out-of-bag score as free validation |
| 4 | Say why a forest's large train/CV gap is *not* the alarm it was yesterday |
| 5 | Spot the trap in `feature_importances_`, and use permutation importance instead |
| 6 | Name three situations where a forest is the wrong choice |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings
warnings.simplefilter("ignore")

from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold,
                                     cross_val_score, cross_validate,
                                     GridSearchCV, train_test_split)

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

CV = StratifiedKFold(5, shuffle=True, random_state=0)
print("ready")

---
# 0 · The data

Same scholarship committee as yesterday, but a harder, more realistic file: **1200
decisions** and **ten columns** instead of four.

Three of those ten columns — `siblings`, `distance_km`, `club_hours` — have **nothing
to do with the committee's rule**. They are there deliberately, because real data files
are full of columns nobody has checked. Keep an eye on what each model does with them.

In [ ]:
rng = np.random.default_rng(4010)
N = 1200

marks      = rng.normal(70, 14, N).clip(35, 100).round(0)
attendance = rng.normal(82, 12, N).clip(40, 100).round(0)
income     = rng.normal(380000, 210000, N).clip(40000, 1500000).round(0)
sport      = rng.binomial(1, 0.25, N).astype(float)
essay      = rng.normal(6.5, 1.8, N).clip(1, 10).round(1)
interview  = rng.normal(6.0, 2.0, N).clip(1, 10).round(1)
siblings   = rng.integers(0, 5, N).astype(float)        # noise
distance   = rng.normal(30, 22, N).clip(1, 150).round(0)  # noise
first_gen  = rng.binomial(1, 0.35, N).astype(float)
club_hours = rng.normal(4, 3, N).clip(0, 20).round(1)   # noise

# the committee's rule: four routes to an award, and an income cap over all of them
acad   = ((marks >= 75) & (attendance >= 80)) | (marks >= 88)
merit  = (essay + interview >= 14.5) & (marks >= 65)
spor   = (sport == 1) & (marks >= 62) & (attendance >= 75)
equity = (first_gen == 1) & (marks >= 68) & (income <= 250000)
rule = (acad | merit | spor | equity) & (income <= 700000)

# ...applied by humans: one decision in ten goes the other way
flip = rng.uniform(0, 1, N) < 0.10
awarded = np.where(flip, 1 - rule.astype(int), rule.astype(int))

sch = pd.DataFrame({"marks": marks, "attendance": attendance, "family_income": income,
                    "sport": sport, "essay": essay, "interview": interview,
                    "siblings": siblings, "distance_km": distance,
                    "first_gen": first_gen, "club_hours": club_hours,
                    "awarded": awarded})
X = sch.drop(columns="awarded")
y = sch["awarded"]
NOISE = ["siblings", "distance_km", "club_hours"]

print(sch.head())
print()
print("rows:", len(sch), " columns:", X.shape[1])
print("awarded:", int(y.sum()), f"({y.mean():.4f})")
print("BASELINE — always guess 'not awarded':", round(1 - y.mean(), 4))

---
# 1 · Where one tree gets to

Before reaching for something new, measure what yesterday's model does here.

In [ ]:
grid = GridSearchCV(DecisionTreeClassifier(random_state=0),
                    {"max_depth": [3, 4, 5, 6, 8, 10, None],
                     "min_samples_leaf": [1, 5, 10, 20]},
                    cv=CV, n_jobs=-1).fit(X, y)
print("tuned tree:", grid.best_params_, "->", round(grid.best_score_, 4))
print()
for name, model in [
        ("one tree, unlimited", DecisionTreeClassifier(random_state=0)),
        ("one tree, tuned", DecisionTreeClassifier(**grid.best_params_, random_state=0))]:
    r = cross_validate(model, X, y, cv=CV, return_train_score=True)
    print(f"  {name:<22} train {r['train_score'].mean():.4f}   "
          f"CV {r['test_score'].mean():.4f}   gap {r['train_score'].mean() - r['test_score'].mean():+.4f}")
print(f"  {'baseline':<22}                    CV {1 - y.mean():.4f}")

TUNED = dict(max_depth=6, min_samples_leaf=10, random_state=0)

The tuned tree reaches 0.8200 — respectable. But two things should bother you:

1. It took a search over **28 combinations** to find it. Forget to tune and you get
   **0.7492**, barely 18 points above guessing.
2. It is still unstable. Let us measure that properly, the way we started to on Day 39.

## 1.1 · How unstable, exactly?

Refit the tuned tree on a random 90% of the rows, five times. Then ask all five about the
same 200 applicants, and count how many get **two different answers** depending only on
which rows happened to be left out.

In [ ]:
Xq = X.iloc[:200]          # the 200 applicants we will keep asking about

def instability(make_model, n_repeats=5):
    preds = []
    for s in range(n_repeats):
        idx = np.random.default_rng(s).choice(len(X), int(0.9 * len(X)), replace=False)
        preds.append(make_model().fit(X.iloc[idx], y.iloc[idx]).predict(Xq))
    preds = np.array(preds)
    return int((preds.min(axis=0) != preds.max(axis=0)).sum())

flips_tree = instability(lambda: DecisionTreeClassifier(**TUNED))
print(f"one tuned tree: {flips_tree} of 200 applicants got two different answers "
      f"({flips_tree / 2:.1f}%)")

**One applicant in five.** Not because anything about them changed — because we shuffled
the training data.

Remember why, from Day 39: the tree is built greedily. Two candidate questions can be
almost exactly tied on impurity, the winner takes the whole branch beneath it, and a
handful of rows decides which one wins. Everything downstream then changes.

> If a model gives one in five people a different answer when you reshuffle the rows,
> it is partly telling you about the shuffle rather than about the people.

---
# 2 · The idea: grow many trees and vote

If one tree is a coin-flip away from a different answer, grow a hundred and count.

But there is an obvious problem first: fit the same algorithm to the same 1200 rows a
hundred times and you get **the same tree a hundred times**. A hundred identical opinions
is worth exactly one opinion.

So each tree must be grown on its own, slightly different, copy of the data.

## 2.1 · Bootstrap sampling

Draw 1200 rows from the 1200 rows **with replacement**. Some rows get drawn twice or
three times; some are never drawn at all.

In [ ]:
idx = np.random.default_rng(7).integers(0, 1200, 1200)
counts = np.bincount(idx, minlength=1200)

print("one tree's bootstrap sample of 1200 rows:")
for label, n in [("never drawn (left out)", (counts == 0).sum()),
                 ("drawn once",            (counts == 1).sum()),
                 ("drawn twice",           (counts == 2).sum()),
                 ("drawn 3 times",         (counts == 3).sum()),
                 ("drawn 4+ times",        (counts >= 4).sum())]:
    print(f"  {label:<24}{n:>5}")
print()
print(f"rows this tree actually sees: {(counts > 0).sum()} of 1200 "
      f"= {(counts > 0).mean():.4f}")

About **63%**, and that is not a coincidence of this seed. The probability that a
particular row is missed on one draw is $1 - 1/1200$, and it must be missed 1200 times
in a row:

$$P(\text{row is left out}) = \left(1 - \tfrac{1}{n}\right)^{n} \;\longrightarrow\; \tfrac{1}{e} \approx 0.368$$

So each tree sees about $1 - 1/e = 0.6321$ of the data, every time, for any reasonably
large $n$.

In [ ]:
fr = [len(set(np.random.default_rng(s).integers(0, 1200, 1200))) / 1200
      for s in range(400)]
print(f"over 400 bootstrap samples: mean {np.mean(fr):.4f}, "
      f"min {min(fr):.4f}, max {max(fr):.4f}")
print(f"theory 1 - 1/e            : {1 - 1 / np.e:.4f}")

plt.figure(figsize=(8, 3.6))
plt.hist(fr, bins=26, color="#1F9E86")
plt.axvline(1 - 1/np.e, color="#F0A530", lw=3, ls="--")
plt.xlabel("fraction of rows the tree actually sees")
plt.ylabel("count"); plt.title("always about 63%, never all of it")
plt.tight_layout(); plt.show()

## 2.2 · Growing the forest

Two lines, and no tuning.

In [ ]:
forest = RandomForestClassifier(n_estimators=100, random_state=0)
forest.fit(X, y)

print("CV scores, identical folds:")
print(f"  baseline            {1 - y.mean():.4f}")
print(f"  one tree, tuned     {cross_val_score(DecisionTreeClassifier(**TUNED), X, y, cv=CV).mean():.4f}")
print(f"  forest of 100       {cross_val_score(forest, X, y, cv=CV).mean():.4f}")

**0.8383 against 0.8200**, with no `GridSearchCV` and no thought at all.

Note there is still no `StandardScaler` anywhere — a forest is made of trees, and trees
compare rather than measure distance, so units remain irrelevant.

## 2.3 · How many trees?

In [ ]:
ns = [1, 2, 3, 5, 10, 25, 50, 100, 200, 300, 500]
scores = [cross_val_score(RandomForestClassifier(n_estimators=n, random_state=0),
                          X, y, cv=CV).mean() for n in ns]
for n, s in zip(ns, scores):
    print(f"  {n:>4} trees   {s:.4f}")

plt.figure(figsize=(9, 4))
plt.plot(range(len(ns)), scores, "o-", lw=2.5, ms=8, color="#1F9E86")
plt.axhline(0.8200, ls="--", color="#8268B5", lw=2, label="one tuned tree 0.8200")
plt.axhline(1 - y.mean(), ls=":", color="#7B8AA1", lw=2, label="baseline")
plt.xticks(range(len(ns)), [str(n) for n in ns])
plt.xlabel("n_estimators"); plt.ylabel("5-fold CV accuracy")
plt.legend(); plt.title("more trees never hurts — it just stops helping")
plt.tight_layout(); plt.show()

The curve rises steeply and then **flattens by about 50 trees**. Past that you are buying
runtime, not accuracy.

This is worth contrasting sharply with `max_depth` yesterday. Depth had a **sweet spot**:
too little underfit, too much overfit. `n_estimators` has no sweet spot — more is never
worse, it just stops paying. Those are two completely different kinds of knob, and
confusing them is a common mistake.

## 2.4 · Does it fix the instability?

This is the real question. Repeat the exact test from section 1.1.

In [ ]:
flips_forest = instability(lambda: RandomForestClassifier(n_estimators=100, random_state=0))

print(f"one tuned tree : {flips_tree:>3} of 200 changed answer  ({flips_tree / 2:.1f}%)")
print(f"forest of 100  : {flips_forest:>3} of 200 changed answer  ({flips_forest / 2:.1f}%)")
print()
print(f"the forest is {flips_tree / flips_forest:.1f}x more stable on this test")

**From 41 applicants to 8.**

This matters more than the 1.8 points of accuracy. A scholarship committee can live with
a model that is 82% accurate. It cannot live with a model that changes its mind about one
applicant in five every time the data is refreshed.

---
# 3 · The second dice: random columns

Bootstrap sampling alone gives you **bagging** — bootstrap aggregating. A random forest
adds a second source of randomness:

> At **every single split**, the tree is shown only `max_features` of the columns, chosen
> at random, and must pick its question from those.

Why bother? Because `marks` is by far the strongest column here. Give every tree all ten
columns and every tree picks `marks` first, and splits it at nearly the same place. The
trees end up correlated — and averaging correlated things barely helps.

In [ ]:
print("bagging  (all columns at every split):",
      round(cross_val_score(BaggingClassifier(DecisionTreeClassifier(random_state=0),
                                              n_estimators=200, random_state=0),
                            X, y, cv=CV).mean(), 4))
print("forest   (3 random columns per split):",
      round(cross_val_score(RandomForestClassifier(n_estimators=200, random_state=0),
                            X, y, cv=CV).mean(), 4))

On *this* data bagging actually scores a shade higher. Before concluding anything from
that, measure it properly — a single 5-fold run is one sample.

## 3.1 · Tuning max_features, honestly

In [ ]:
RCV = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)  # 50 fits each

rows = []
for mf in [1, 2, 3, 5, 7, None]:
    s = cross_val_score(RandomForestClassifier(n_estimators=200, max_features=mf,
                                               random_state=0), X, y, cv=RCV, n_jobs=-1)
    rows.append({"max_features": "all 10" if mf is None else mf,
                 "mean": round(s.mean(), 4), "sd": round(s.std(), 4)})
print(pd.DataFrame(rows).to_string(index=False))
print("\n(the default for classification is sqrt(10) = 3)")

Read the `sd` column before the `mean` column. The standard deviation across folds is
about **0.019**, and the entire spread from `max_features=3` to `max_features=7` is about
**0.009** — half the noise.

> **So the honest conclusion is: only `max_features=1` is clearly worse. Between 3 and 10,
> nothing is established.** It is worth putting `max_features` in a grid search, but do
> not expect it to rescue a model.

This is the same discipline as Day 37 and Day 38: a difference smaller than the spread
between folds is not a finding.

---
# 4 · Out-of-bag scoring: validation for free

Each tree skipped about 37% of the rows. So for any given row, roughly a third of the
forest has never seen it.

Let only *those* trees vote on it, and you get a prediction on genuinely held-out data —
without splitting anything, and without fitting a single extra model.

In [ ]:
oob_forest = RandomForestClassifier(n_estimators=300, oob_score=True,
                                    random_state=0).fit(X, y)

print(f"training accuracy   {oob_forest.score(X, y):.4f}   <- it has seen every row")
print(f"OOB score           {oob_forest.oob_score_:.4f}   <- free")
print(f"5-fold CV           "
      f"{cross_val_score(RandomForestClassifier(n_estimators=300, random_state=0), X, y, cv=CV).mean():.4f}"
      f"   <- 5 extra forests fitted")

OOB is a little optimistic here (0.8475 against 0.8367), which is typical — each row is
judged by only a third of the forest, and OOB uses all 1200 rows for both roles.

**Use it to iterate quickly. Report the cross-validated number.**

OOB needs `bootstrap=True` (the default) and enough trees that every row is out-of-bag
for a decent number of them. With 20 trees it is unreliable; with 300 it is fine.

## 4.1 · Why the forest's huge train/CV gap is not the alarm it was yesterday

In [ ]:
for name, model in [
        ("one tree, unlimited", DecisionTreeClassifier(random_state=0)),
        ("one tree, tuned",     DecisionTreeClassifier(**TUNED)),
        ("forest of 100",       RandomForestClassifier(n_estimators=100, random_state=0))]:
    r = cross_validate(model, X, y, cv=CV, return_train_score=True)
    print(f"  {name:<22} train {r['train_score'].mean():.4f}   CV {r['test_score'].mean():.4f}"
          f"   gap {r['train_score'].mean() - r['test_score'].mean():+.4f}")

The forest scores a perfect **1.0000** on its training rows and shows a gap of
**+0.1617**. On Day 39 a gap like that was the signature of a model that had memorised
its data. Here it is not a problem, and it is worth being precise about why.

- Each tree is grown to full depth on its own bootstrap sample, so each one memorises
  *that* sample. Between the hundred of them, the forest has an answer written down for
  essentially every training row. **A large training gap is guaranteed by construction.**
- What was ever actually at stake is the honest score, and the forest's is the best on
  this data.

So the rule from Day 39 needs sharpening rather than discarding:

> A train/CV gap is a **diagnostic, not a target**. It tells you to go and look. For a
> single tree, looking reveals a real problem. For a forest, looking reveals the
> mechanism working as designed.

Which is also why you should **not** reach for `max_depth` on a forest. Try it:

In [ ]:
for depth in [3, 5, 8, None]:
    r = cross_validate(RandomForestClassifier(n_estimators=100, max_depth=depth,
                                              random_state=0), X, y, cv=CV,
                       return_train_score=True)
    print(f"  max_depth={str(depth):<5} train {r['train_score'].mean():.4f}"
          f"   CV {r['test_score'].mean():.4f}"
          f"   gap {r['train_score'].mean() - r['test_score'].mean():+.4f}")

Pruning closes the gap, and the CV score falls with it — **except** at `max_depth=8`,
which comes out at 0.8433, apparently *above* unlimited's 0.8383.

Do not accept that at face value. It is one 5-fold split, and we have been told all week
what a single split is worth. Measure it again properly.

In [ ]:
RCV_D = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)
scores = {}
for depth in [3, 5, 8, 12, None]:
    scores[depth] = cross_val_score(
        RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=0),
        X, y, cv=RCV_D, n_jobs=-1)
    print(f"  max_depth={str(depth):<5} {scores[depth].mean():.4f}  sd {scores[depth].std():.4f}")

d8 = scores[8] - scores[None]
print(f"\n  depth 8 minus unlimited: {d8.mean():+.4f}"
      f"   — depth 8 won {(d8 > 0).sum()} of {len(d8)} folds")

Over 50 folds the picture is clear and the fluke disappears: depth 8 is **0.0018 worse**
on average and wins only **17 of 50** folds.

So the real conclusion is:

- Shallow caps **clearly hurt**: 0.7918 at depth 3, 0.8141 at depth 5, against 0.8349
  unlimited.
- Deep caps make **no difference you can establish**.
- **Closing the gap never bought any accuracy**, at any depth.

Two lessons for the price of one: do not prune a forest, and do not trust a single split
even when it tells you something you might have wanted to hear.

---
# 5 · What you gave up

The forest is more accurate and far more stable. Three things got worse.

## 5.1 · You cannot read it

In [ ]:
one = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X, y)
print("A depth-3 tree, in full:\n")
print(export_text(one, feature_names=list(X.columns), decimals=0))

leaves = sum(t.get_n_leaves() for t in forest.estimators_)
print(f"The forest: {len(forest.estimators_)} trees, {leaves} leaves between them.")
print("There is no export_text for a forest, and there could not usefully be one.")

That is a real trade, not a technicality. In hiring, lending, admissions and medicine,
*"why?"* is often a legal requirement, and *"the forest voted 57 to 43"* is not an answer
you can put in a letter.

What you **do** get back is the vote share, which a single tree cannot give you at all.

In [ ]:
proba = forest.predict_proba(X)[:, 1]
print("how divided is the forest, across all 1200 applicants?")
for lo, hi in [(0, .1), (.1, .3), (.3, .5), (.5, .7), (.7, .9), (.9, 1.01)]:
    n = ((proba >= lo) & (proba < hi)).sum()
    print(f"  {lo:.1f} - {min(hi, 1.0):.1f}  {'#' * int(n / 12):<38} {n:>4}")

i = int(np.argmin(np.abs(proba - 0.5)))
print(f"\nthe most divided applicant is row {i}: "
      f"{int(round(proba[i] * 100))} of 100 trees voted AWARD")

Most applicants are near 0 or near 1 — the forest is confident and nearly unanimous. A
small group sits in the middle, and those are exactly the files a human should look at.
**"57 of 100" is a genuinely different thing to say than "98 of 100",** and a single tree
cannot distinguish them.

## 5.2 · It is far slower

In [ ]:
specs = [("one tree", DecisionTreeClassifier(**TUNED)),
         ("forest, 100", RandomForestClassifier(n_estimators=100, random_state=0)),
         ("forest, 500", RandomForestClassifier(n_estimators=500, random_state=0))]

print(f"{'model':<14}{'fit (ms)':>12}{'predict (ms)':>15}")
base_f = base_p = None
for name, m in specs:
    f_ms = min(( (lambda t0: (m.fit(X, y), time.perf_counter() - t0)[1])(time.perf_counter())
                 for _ in range(5) )) * 1000
    p_ms = min(( (lambda t0: ([m.predict(X) for _ in range(10)],
                              (time.perf_counter() - t0) / 10)[1])(time.perf_counter())
                 for _ in range(5) )) * 1000
    base_f, base_p = (base_f or f_ms), (base_p or p_ms)
    print(f"{name:<14}{f_ms:>9.1f} ({f_ms/base_f:>3.0f}x){p_ms:>11.2f} ({p_ms/base_p:>3.0f}x)")

Irrelevant when you fit once on 1200 rows. It decides the design when you retrain nightly
on ten million, or have to answer in a few milliseconds inside an app.

You can buy some of it back with **`n_jobs=-1`**: the trees are completely independent,
so they can be grown on every core at once. (Boosting, tomorrow, cannot do this — each
tree there depends on the one before it.)

## 5.3 · The importances it gives you are misleading

This is the subtle one, and the most important thing in today's lesson.

Three of our ten columns are pure noise. We built them that way. A trustworthy importance
ranking should put all three at zero.

In [ ]:
f300 = RandomForestClassifier(n_estimators=300, random_state=0).fit(X, y)
imp = pd.Series(f300.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.round(4).to_string())
print(f"\nthe three NOISE columns are handed {imp[NOISE].sum():.4f} between them")
print("club_hours — pure noise — ranks", list(imp.index).index("club_hours") + 1, "of 10")

`club_hours` is random data with no relationship to anything, and the forest ranks it
**sixth of ten**, above `sport` and `first_gen`, both of which are genuinely in the rule.

Three reasons this happens, all worth understanding:

1. **It is computed on the training rows.** A noise column will, by luck, separate a few
   training rows — and that counts as impurity removed, even though the luck will not
   repeat on new data.
2. **`max_features` makes it worse.** When the strong columns are not among the three
   offered at a split, the tree must choose from what it has. Junk gets picked, and
   credited.
3. **Columns with many distinct values collect more credit**, because they get more
   chances to be chosen. `distance_km` offers 85 possible thresholds; `sport` offers one.

### The fix: permutation importance

Ask the question you actually meant: *if I destroy this column, does the model get worse?*
Shuffle one column on **held-out** data, re-measure accuracy, and see how far it falls.

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
fm = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xtr, ytr)
pi = permutation_importance(fm, Xte, yte, n_repeats=30, random_state=0)

out = pd.DataFrame({"impurity (default)": fm.feature_importances_,
                    "permutation": pi.importances_mean,
                    "perm sd": pi.importances_std}, index=X.columns)
out["real?"] = ["yes" if c not in NOISE else "NOISE" for c in X.columns]
print(out.round(4).sort_values("permutation", ascending=False).to_string())
print(f"\nnoise columns — impurity {out.loc[NOISE, 'impurity (default)'].sum():.4f}"
      f"   permutation {out.loc[NOISE, 'permutation'].sum():+.4f}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.4))
col = lambda idx: ["#E05C45" if c in NOISE else "#3E7CB1" for c in idx]

s1 = pd.Series(fm.feature_importances_, index=X.columns).sort_values()
a1.barh(s1.index, s1.values, color=col(s1.index))
a1.set_title("feature_importances_  (the default)"); a1.set_xlabel("importance")

s2 = pd.Series(pi.importances_mean, index=X.columns).sort_values()
a2.barh(s2.index, s2.values, xerr=pd.Series(pi.importances_std, index=X.columns)[s2.index],
        color=["#E05C45" if c in NOISE else "#1F9E86" for c in s2.index])
a2.axvline(0, color="#131F33", lw=1.4)
a2.set_title("permutation importance"); a2.set_xlabel("drop in accuracy when shuffled")
plt.tight_layout(); plt.show()

Permutation importance puts all three junk columns at **essentially zero, and two of them
slightly negative** — shuffling them sometimes *helped*, which is exactly what random
noise should do.

> **Never rank columns with `feature_importances_` alone.** Shuffle and re-measure. And
> even then, importance is association, not cause: it tells you what the model leaned on,
> not what drives the outcome in the world.

---
# 6 · When a forest is the wrong answer

## 6.1 · When the truth is smooth

Back to Day 39's exam data, where passing depends smoothly on hours studied and previous
score — a diagonal, not a set of thresholds.

In [ ]:
r2 = np.random.default_rng(777)
M = 900
hours = r2.normal(12, 4, M).clip(0, 30).round(1)
prev  = r2.normal(60, 15, M).clip(10, 100).round(0)
passed = (r2.uniform(0, 1, M) <
          1 / (1 + np.exp(-(-7.5 + 0.30 * hours + 0.075 * prev)))).astype(int)
XL = pd.DataFrame({"hours": hours, "prev_score": prev})
yL = pd.Series(passed)

print(f"  {'baseline':<22}{max(yL.mean(), 1 - yL.mean()):.4f}")
for name, m in [("logistic regression", make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))),
                ("one tree (depth 4)",  DecisionTreeClassifier(max_depth=4, random_state=0)),
                ("forest (300 trees)",  RandomForestClassifier(n_estimators=300, random_state=0))]:
    print(f"  {name:<22}{cross_val_score(m, XL, yL, cv=CV).mean():.4f}")

**The forest finishes last — below the single tree it was built from.**

A hundred axis-aligned staircases averaged together is a smoother staircase. It never
becomes a diagonal. Ensembling fixes *variance*; it cannot fix a model whose shape is
wrong for the problem.

## 6.2 · The summary table

In [ ]:
contenders = [
    ("baseline (majority)", None),
    ("k-NN (k=15)",         make_pipeline(StandardScaler(), KNeighborsClassifier(15))),
    ("logistic regression", make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))),
    ("one tree (unlimited)", DecisionTreeClassifier(random_state=0)),
    ("one tree (tuned)",    DecisionTreeClassifier(**TUNED)),
    ("random forest (100)", RandomForestClassifier(n_estimators=100, random_state=0)),
]
for name, m in contenders:
    s = 1 - y.mean() if m is None else cross_val_score(m, X, y, cv=CV).mean()
    print(f"  {name:<24}{s:.4f}")

## 6.3 · Is the forest's win real?

0.8383 against 0.8200 is 1.8 points, from a single 5-fold split. Day 38 taught us not to
trust a difference smaller than the noise. So measure it the right way: **the same folds,
both models, many repeats, and count how often the forest wins.**

In [ ]:
t_scores = cross_val_score(DecisionTreeClassifier(**TUNED), X, y, cv=RCV, n_jobs=-1)
f_scores = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=0),
                           X, y, cv=RCV, n_jobs=-1)
diff = f_scores - t_scores

print(f"over {len(diff)} folds (5-fold x 10 repeats):")
print(f"  one tuned tree   {t_scores.mean():.4f}  (sd {t_scores.std():.4f})")
print(f"  forest of 100    {f_scores.mean():.4f}  (sd {f_scores.std():.4f})")
print()
print(f"  forest minus tree: {diff.mean():+.4f}  (sd {diff.std():.4f})")
print(f"  the forest won in {(diff > 0).sum()} of {len(diff)} folds")

**+0.0308 on average, winning 42 of 50 folds.** That is an established difference — not
because the average is positive, but because it holds up again and again.

Note also that the single 5-fold run flattered the *tree* (0.8200 against its 50-fold
average of 0.8041). Any single split is one sample.

---
# 7 · Summary

| | |
|---|---|
| **The idea** | Grow many trees on bootstrap samples, show each split only a random subset of columns, and answer by vote. |
| **Why it works** | Each tree is wrong in its own way; averaging cancels the idiosyncrasies and keeps the signal. |
| **The real win** | Stability. Applicants whose answer changed across refits: **41 of 200 → 8 of 200**. |
| **Tuning** | Almost none. `n_estimators` as high as you can afford (flat by ~50). Do **not** prune — averaging does that job. |
| **OOB** | Each tree skips ~37% of rows, so the forest grades itself for free. 0.8475 here. |
| **The train gap** | 1.0000 on train is expected and harmless. The gap is a diagnostic, never a target. |
| **The costs** | Unreadable, tens of times slower, and `feature_importances_` ranked a pure-noise column 6th of 10. |
| **When not to** | Smooth truths, when you must explain each decision, and when prediction must be very fast. |

### The numbers from today

| quantity | value |
|---|---|
| baseline | 0.5708 |
| one tree, unlimited / tuned | 0.7492 / 0.8200 |
| logistic regression / k-NN | 0.7833 / 0.7683 |
| random forest, 100 trees | **0.8383** |
| forest advantage over 50 folds | +0.0308, winning 42 of 50 |
| rows one tree sees | 63.2% (= 1 − 1/e) |
| OOB score, 300 trees | 0.8475 |
| importance given to 3 noise columns | impurity 0.1831, permutation −0.0117 |
| instability, tree → forest | 41/200 → 8/200 |

---

### Next: Day 41 — Gradient Boosting

A forest grows a hundred trees **in parallel**, all equals, and averages them. Tomorrow we
grow them **one at a time**, each new tree trained specifically on what the previous ones
got wrong. It is usually more accurate still — and far easier to break.